# 01. TF-IDF Model Training

**Goal:**  
Train our custom Multinomial Naive Bayes model using TF-IDF feature weighting (`TfidfVectorizer`) combined with bigrams to evaluate smooth document-frequency scaling against raw counts.

**Pipeline Overview:**
1. **Data Ingestion:** Load `data/processed/train_cleaned.csv`.
2. **TF-IDF Extraction:** Transform text using `fit_transform_tfidf_train`.
3. **Model Fitting:** Fit custom `MultinomialNaiveBayes` on float feature weights.
4. **Artifact Storage:** Save binaries inside `models/tfidf_version/`.

In [3]:
import sys
import os
import pickle
import pandas as pd
import numpy as np

# Locate custom modules in src/
sys.path.append(os.path.abspath("../../src"))

from preprocessing import fit_transform_tfidf_train
from naive_bayes import MultinomialNaiveBayes

# Load clean training dataset
df_train = pd.read_csv("../../data/processed/train_cleaned.csv").dropna(subset=['tweet_content'])
print(f"Loaded training samples: {len(df_train)}")

Loaded training samples: 57297


## 2. TF-IDF Feature Extraction

We extract unigrams and bigrams using TF-IDF. Notice we raise `max_df` back to `0.50` to let TF-IDF smoothly scale down frequent terms without forcibly cutting off vocabulary.

In [4]:
X_train_text = df_train['tweet_content']
y_train = df_train['sentiment'].values

# Fit TF-IDF Vectorizer
vectorizer, X_train_sparse = fit_transform_tfidf_train(
    X_train_text,
    min_df=10,
    max_df=0.50,          # Safer ceiling; TF-IDF handles term weighting
    ngram_range=(1, 2)
)

n_samples, n_features = X_train_sparse.shape
print(f"TF-IDF Feature Matrix Shape: {n_samples} rows x {n_features} features")
print(f"Matrix Sparsity: {100 * (1 - X_train_sparse.nnz / (n_samples * n_features)):.2f}%")

TF-IDF Feature Matrix Shape: 57297 rows x 20086 features
Matrix Sparsity: 99.88%


## 3. Model Fitting & Serialization

We train the model and save artifacts into a dedicated `models/tfidf_version/` directory.

In [5]:
# 1. Fit Model
model = MultinomialNaiveBayes(alpha=1.0)
model.fit(X_train_sparse, y_train)
print(f"Model successfully fitted on classes: {model.classes_}")

# 2. Create directory if not present
tfidf_dir = "../../models/Tf-Idf version"
os.makedirs(tfidf_dir, exist_ok=True)

# 3. Save Model Binary
model_path = os.path.join(tfidf_dir, "tfidf_naive_bayes_model.pkl")
model.save_model(model_path)
print(f"Saved TF-IDF model state to: {model_path}")

# 4. Save Vectorizer Binary
vectorizer_path = os.path.join(tfidf_dir, "tfidf_vectorizer.pkl")
with open(vectorizer_path, "wb") as f:
    pickle.dump(vectorizer, f)
print(f"Saved TF-IDF vectorizer to: {vectorizer_path}")

Model successfully fitted on classes: ['Negative' 'Neutral' 'Positive']
Saved TF-IDF model state to: ../../models/Tf-Idf version\tfidf_naive_bayes_model.pkl
Saved TF-IDF vectorizer to: ../../models/Tf-Idf version\tfidf_vectorizer.pkl
